# درس ۰۱ - معرفی عامل‌های هوش مصنوعی

به اولین درس در دوره **عامل‌های هوش مصنوعی برای مبتدیان** خوش آمدید!

یک **عامل هوش مصنوعی** برنامه‌ای است که از یک مدل زبان بزرگ (LLM) به عنوان موتور استدلال خود استفاده می‌کند و می‌تواند در دنیای واقعی *اقداماتی* انجام دهد — فراخوانی APIها، پرس‌وجوی پایگاه‌های داده، یا اجرای کد — تا به نیابت از یک کاربر هدفی را تحقق بخشد.

در این نوت‌بوک شما اولین عامل خود را خواهید ساخت: یک **عامل مسافرتی** که مقاصد تعطیلات را پیشنهاد می‌دهد. در طول مسیر یاد خواهید گرفت چگونه:

1. به سرویس عامل Azure AI Foundry با استفاده از **چارچوب عامل مایکروسافت** متصل شوید.
2. به عامل یک **ابزار** بدهید — یک تابع ساده پایتون که می‌تواند آن را فراخوانی کند.
3. عامل را اجرا کنید و پاسخ آن را بررسی کنید.
4. پاسخ عامل را به صورت توکن به توکن پخش کنید.


## راه‌اندازی

قبل از اجرای این دفترچه، مطمئن شوید که:

1. **یک پروژه Azure AI Foundry** با یک مدل چت مستقر شده (مثلاً `gpt-4o-mini`) دارید.
2. **با Azure CLI وارد شده‌اید** — دستور `az login` را در ترمینال خود اجرا کنید.
3. **متغیرهای محیطی مورد نیاز را تنظیم کرده‌اید:**
   - `AZURE_AI_PROJECT_ENDPOINT` — نقطه‌پایان پروژه Azure AI Foundry شما.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — نام مدل مستقر شده شما.

سلول زیر بسته‌های پایتونی مورد نیاز شما را نصب می‌کند.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## ایجاد اولین نماینده شما

یک نماینده به دو چیز نیاز دارد:

- **دستورات** که به آن می‌گویند *که کیست* و *چگونه رفتار کند* (یک پرامپت سیستم).
- **ابزارها** — توابع پایتونی که با `@tool` تزئین شده‌اند و نماینده می‌تواند از آن‌ها برای دریافت اطلاعات یا انجام کارها استفاده کند.

در زیر یک ابزار ساده تعریف کرده‌ایم که فهرستی از مقاصد محبوب تعطیلات را بازمی‌گرداند. هنگامی که کاربر درخواست توصیه‌های سفر کند، نماینده از این ابزار استفاده خواهد کرد.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## پاسخ‌های جریان‌دار

برای تجربه‌ای تعاملی‌تر می‌توانید پاسخ عامل را به‌صورت **جریان‌دار** دریافت کنید. به جای انتظار برای دریافت پاسخ کامل، عامل بخش‌های متنی را همان‌طور که تولید می‌شوند ارائه می‌دهد. این ویژگی به ویژه در رابط‌های چت که می‌خواهید خروجی را به صورت زنده نمایش دهید، مفید است.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## خلاصه

در این درس یاد گرفتید چگونه:

- **یک provider ایجاد کنید** که به سرویس Azure AI Foundry Agent از طریق `AzureAIProjectAgentProvider` متصل می‌شود.
- **یک ابزار تعریف کنید** با استفاده از دکوراتور `@tool` تا عوامل بتوانند توابع پایتون شما را فراخوانی کنند.
- **عامل را اجرا کنید** با پیام کاربر و پاسخ آن را چاپ کنید.
- **پاسخ‌ها را به صورت استریم دریافت کنید** برای خروجی زمان واقعی.

در درس بعدی چارچوب‌های عاملی را عمیق‌تر بررسی خواهیم کرد و یاد می‌گیریم چگونه ابزارهای قدرتمندتر و قابلیت‌های استدلال چندمرحله‌ای را به عوامل بدهیم.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**سلب مسئولیت**:  
این سند با استفاده از خدمات ترجمه هوش مصنوعی [Co-op Translator](https://github.com/Azure/co-op-translator) ترجمه شده است. در حالی که ما تلاش می‌کنیم دقت را حفظ کنیم، لطفاً توجه داشته باشید که ترجمه‌های خودکار ممکن است حاوی خطاها یا نواقص باشند. سند اصلی به زبان مادری خود باید به عنوان منبع معتبر در نظر گرفته شود. برای اطلاعات حیاتی، ترجمه حرفه‌ای انسانی توصیه می‌شود. ما مسئول هیچ گونه سوء تفاهم یا تفسیر نادرستی که از استفاده این ترجمه ناشی شود، نمی‌باشیم.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
